In [1]:
# ============================================================
# Loan Default Prediction – Final Feature Engineering Pipeline
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------
DATA_PATH = "../data/raw/Loan_Default.csv"   # change path if needed
df = pd.read_csv(DATA_PATH)

print("Initial Shape:", df.shape)

Initial Shape: (255347, 18)


In [3]:
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [ ]:
cols_to_ignore = ['LoanID']
target_col = 'Default'
label_encode_cols = ['HasMortgage','HasDependents','HasCoSigner']
one_hot_encode_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']

In [4]:
df.columns

Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='object')

In [5]:
# ------------------------------------------------------------
# 2. BASIC CLEANING
# ------------------------------------------------------------

# Remove duplicates
df.drop_duplicates(inplace=True)

In [6]:

# Drop ID-like columns
df.drop(columns=['LoanID'], inplace=True, errors="ignore")

print("After cleaning:", df.shape)

After cleaning: (255347, 17)


In [7]:
# ------------------------------------------------------------
# 3. TARGET VARIABLE
# ------------------------------------------------------------
TARGET = "Default"
X = df.drop(columns=[TARGET])
y = df[TARGET]

In [8]:
# ------------------------------------------------------------
# 4. FEATURE ENGINEERING (DERIVED FEATURES)
# ------------------------------------------------------------

# Financial pressure indicators
X["loan_interest_burden"] = X["LoanAmount"] * X["InterestRate"]
X["loan_term_pressure"] = X["LoanAmount"] / (X["LoanTerm"] + 1)

In [9]:
[col for col in df.columns if col not in ['HasCoSigner','LoanPurpose','HasDependents', 
                        'HasMortgage','MaritalStatus', 'EmploymentType', 
                        'Education']]

['Age',
 'Income',
 'LoanAmount',
 'CreditScore',
 'MonthsEmployed',
 'NumCreditLines',
 'InterestRate',
 'LoanTerm',
 'DTIRatio',
 'Default']

In [10]:
# ------------------------------------------------------------
# 5. SELECT FINAL FEATURES
# ------------------------------------------------------------

numerical_features = ['Age',
 'Income',
 'LoanAmount',
 'CreditScore',
 'MonthsEmployed',
 'NumCreditLines',
 'InterestRate',
 'LoanTerm',
 'DTIRatio']

categorical_features = ['HasCoSigner','LoanPurpose','HasDependents', 
                        'HasMortgage','MaritalStatus', 'EmploymentType', 
                        'Education']

X = X[numerical_features + categorical_features]

In [11]:
X['LoanPurpose'].unique()

array(['Other', 'Auto', 'Business', 'Home', 'Education'], dtype=object)

In [12]:
# ------------------------------------------------------------
# 6. PREPROCESSING PIPELINES
# ------------------------------------------------------------

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    # ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)


In [13]:


# ------------------------------------------------------------
# 7. MODEL PIPELINE
# ------------------------------------------------------------
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier 

lr = LogisticRegression(max_iter=1000)
xgb = XGBClassifier(eval_metric="logloss")
lgbm = LGBMClassifier()

model_pipeline = Pipeline(steps=[("preprocessor", preprocessor),
    ("model", xgb)
])


In [14]:
# ------------------------------------------------------------
# 8. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,random_state=42,
    stratify=y
)


In [15]:
# ------------------------------------------------------------
# 9. TRAIN MODEL
# ------------------------------------------------------------

model_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
# ------------------------------------------------------------
# 10. EVALUATION
# ------------------------------------------------------------

y_pred = model_pipeline.predict(X_test)
print("\nMODEL PERFORMANCE\n")
print(classification_report(y_test, y_pred))

# Predict probability for positive class (class = 1)
y_pred_lr = model_pipeline.predict_proba(X_test)[:, 1]
# Calculate AUC-ROC score
from sklearn.metrics import roc_auc_score
auc_lr = roc_auc_score(y_test, y_pred_lr)
print("Logistic Regression AUC-ROC:", auc_lr)


MODEL PERFORMANCE

              precision    recall  f1-score   support

           0       0.89      0.99      0.94     45139
           1       0.56      0.08      0.15      5931

    accuracy                           0.89     51070
   macro avg       0.72      0.54      0.54     51070
weighted avg       0.85      0.89      0.85     51070

Logistic Regression AUC-ROC: 0.7431500866640566


In [49]:
# ------------------------------------------------------------
# 11. SAVE PIPELINE
# ------------------------------------------------------------

joblib.dump(model_pipeline, "loan_default_model.pkl")
joblib.dump(X.columns.tolist(), "model_features.pkl")

print("\nModel and feature list saved successfully!")


Model and feature list saved successfully!


In [ ]:
# ============================================================
# 12. INFERENCE FUNCTION (RAW USER INPUT)
# ============================================================

def predict_loan_default(raw_input: dict):
    """
    raw_input example:
    {
        "loan_amount": 2500000,
        "rate_of_interest": 9.5,
        "term": 240,
        "LTV": 75,
        "Upfront_charges": 15000,
        "Credit_Worthiness": 720,
        "loan_type": "Home Loan",
        "Security_Type": "Direct",
        "LoanPurpose": "Purchase",
        "open_credit": "No",
        "business_or_commercial": "No",
        "approve_in_advance": "Yes",
        "Neg_ammortization": "No",
    }
    """

    model = joblib.load("loan_default_model.pkl")
    feature_cols = joblib.load("model_features.pkl")

    user_df = pd.DataFrame([raw_input])

    # -------- Derive features for inference --------
    user_df["loan_interest_burden"] = user_df["LoanAmount"] * user_df["InterestRate"]
    user_df["loan_term_pressure"] = user_df["loanAmount"] / (user_df["LoanTerm"] + 1)
    
    # Add missing columns
    for col in feature_cols:
        if col not in user_df.columns:
            user_df[col] = np.nan

    user_df = user_df[feature_cols]

    prediction = model.predict(user_df)[0]
    probability = model.predict_proba(user_df)[0][1]

    if probability < 0.3:
        risk = "Low Risk"
        action = "Send payment reminder via SMS/Email"
    elif probability < 0.6:
        risk = "Medium Risk"
        action = "Offer flexible EMI or short-term payment plan"
    else:
        risk = "High Risk"
        action = "Assign to recovery agent and initiate call"

    return {
        "default_probability": float(round(probability*100, 2)),
        "risk_level": risk,
        "recommended_action": action
    }

In [17]:
df.columns

Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education',
       'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents',
       'LoanPurpose', 'HasCoSigner', 'Default'],
      dtype='object')

In [18]:
X['LoanPurpose'].unique()

array(['Other', 'Auto', 'Business', 'Home', 'Education'], dtype=object)

In [19]:
# ------------------------------------------------------------
# 13. SAMPLE INFERENCE TEST
# ------------------------------------------------------------

if __name__ == "__main__":
    sample_input = {
        "LoanAmount": 3000000,
        "InterestRate": 10.2,
        "LoanTerm": 240,
        "LoanPurpose": "Business",
    }
    print("\nInference Output:")
    print(predict_loan_default(sample_input))


Inference Output:


NameError: name 'predict_loan_default' is not defined

In [20]:
df.columns

Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education',
       'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents',
       'LoanPurpose', 'HasCoSigner', 'Default'],
      dtype='object')

In [21]:
df['LoanAmount'].unique()

array([ 50587, 124440, 129188, ..., 105905, 168231, 208294],
      shape=(158729,))

In [25]:
X['InterestRate'].unique()

array([15.23,  4.81, 21.17, ...,  2.46, 19.81,  9.01], shape=(2301,))

In [26]:
X['LoanTerm'].unique()

array([36, 60, 24, 48, 12])

In [22]:
pd.DataFrame([sample_input]).columns

Index(['LoanAmount', 'InterestRate', 'LoanTerm', 'LoanPurpose'], dtype='object')

In [27]:
np.array([105905, 15.23, 48, "Bussiness"]).reshape(1, -1)

array([['105905', '15.23', '48', 'Bussiness']], dtype='<U32')

In [28]:
pd.DataFrame([105905, 15.23, 48, "Bussiness"])

,0
0,105905
1,15.23
2,48
3,Bussiness


In [29]:
pd.DataFrame(np.array([105905, 15.23, 48, "Bussiness"]).reshape(1, -1),columns=['LoanAmount', 'InterestRate', 'LoanTerm','LoanPurpose'])

,LoanAmount,InterestRate,LoanTerm,LoanPurpose
0,105905,15.23,48,Bussiness


In [55]:
import pandas as pd
import numpy as np
import joblib

# Load trained pipeline
model_pipeline = joblib.load("loan_default_model.pkl")

# Extract components
preprocessor = model_pipeline.named_steps["preprocessor"]
model = model_pipeline.named_steps["model"]

# -----------------------------
# Get feature names
# -----------------------------

# Numerical features
num_features = preprocessor.transformers_[0][2]

# Categorical features (after OneHotEncoding)
cat_transformer = preprocessor.transformers_[1][1]
cat_features = cat_transformer.named_steps["encoder"].get_feature_names_out(
    preprocessor.transformers_[1][2]
)

# Combine feature names
all_features = np.concatenate([num_features, cat_features])

# -----------------------------
# Get coefficients
# -----------------------------
coefficients = model.coef_[0]

# Create importance dataframe
feature_importance = pd.DataFrame({
    "Feature": all_features,
    "Coefficient": coefficients,
    "Absolute_Importance": np.abs(coefficients)
})

# Sort by importance
feature_importance = feature_importance.sort_values(
    by="Absolute_Importance", ascending=False
)

print(feature_importance.head(15))

                       Feature  Coefficient  Absolute_Importance
12      Security_Type_Indriect     1.863627             1.863627
13        Security_Type_direct    -1.583911             1.583911
24   Neg_ammortization_neg_amm     0.672679             0.672679
15             loan_purpose_p2     0.580044             0.580044
3                          LTV     0.463433             0.463433
5         loan_interest_burden    -0.418509             0.418509
25   Neg_ammortization_not_neg    -0.392963             0.392963
8         Credit_Worthiness_l2     0.324051             0.324051
4              Upfront_charges    -0.292395             0.292395
0                  loan_amount     0.290954             0.290954
17             loan_purpose_p4    -0.233183             0.233183
10             loan_type_type2     0.221423             0.221423
20  business_or_commercial_b/c     0.221423             0.221423
22         approv_in_adv_nopre     0.219548             0.219548
19             open_credi

In [30]:
import pandas as pd
import numpy as np
import joblib
import statsmodels.api as sm

In [ ]:
# Load trained pipeline
pipeline = joblib.load("loan_default_model.pkl")

# Extract preprocessing step
preprocessor = pipeline.named_steps["preprocessor"]

# Load original dataset again
df = pd.read_csv("Loan_Default.csv")

# Target
y = df["Default"]

# Apply SAME feature engineering as training
X = df.drop(columns=["Default"])

X["loan_interest_burden"] = X["LoanAmount"] * X["InterestRate"]
X["loan_term_pressure"] = X["LoanAmount"] / (X["LoanTerm"] + 1)

# Keep only selected features
feature_cols = joblib.load("model_features.pkl")
X = X[feature_cols]

# Transform features
X_processed = preprocessor.transform(X)

# Add intercept
X_processed = sm.add_constant(X_processed)


In [31]:
# Numerical feature names
num_features = preprocessor.transformers_[0][2]

# Categorical feature names
cat_encoder = preprocessor.transformers_[1][1].named_steps["encoder"]
cat_features = cat_encoder.get_feature_names_out(
    preprocessor.transformers_[1][2]
)

# Final feature list
all_features = np.concatenate([["Intercept"], num_features, cat_features])


In [32]:
OneHotEncoder(handle_unknown="ignore", drop="first")


,categories,'auto'
,drop,'first'
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [33]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])


In [34]:
X_processed = preprocessor.transform(X)
X_processed = sm.add_constant(X_processed)

logit_model = sm.Logit(y, X_processed)
result = logit_model.fit(method="lbfgs", maxiter=200)


C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


In [35]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=1e-5)
X_processed = vt.fit_transform(X_processed)


In [36]:
import numpy as np

print("NaNs:", np.isnan(X_processed).sum())
print("Infs:", np.isinf(X_processed).sum())
print("Rank:", np.linalg.matrix_rank(X_processed))
print("Columns:", X_processed.shape[1])


NaNs: 0
Infs: 0
Rank: 25
Columns: 31


In [37]:
logit_model = sm.Logit(y, X_processed)
result = logit_model.fit(maxiter=100, disp=False)

In [38]:
print("X_processed shape:", X_processed.shape)
print("Params length:", len(result.params))


X_processed shape: (255347, 31)
Params length: 31


In [68]:
# Numerical feature names
num_features = preprocessor.transformers_[0][2]

# Categorical feature names AFTER drop="first"
cat_encoder = preprocessor.transformers_[1][1].named_steps["encoder"]
cat_features = cat_encoder.get_feature_names_out(
    preprocessor.transformers_[1][2]
)

# Combine feature names
feature_names = np.concatenate([num_features, cat_features])

# Add intercept manually
feature_names = np.insert(feature_names, 0, "Intercept")


In [77]:
summary_df = pd.DataFrame({
    "Feature": feature_names[1:],
    "Coefficient": result.params[1:],
    "P_value": result.pvalues
})

summary_df["Significant_0.05"] = summary_df["P_value"] < 0.05
summary_df["Abs_Coefficient"] = summary_df["Coefficient"].abs()

summary_df = summary_df.sort_values(
    by="Abs_Coefficient", ascending=False
)

print(summary_df.head(15))


                       Feature  Coefficient   P_value  Significant_0.05  \
x13            Upfront_charges    12.389096  0.999926             False   
x14       loan_interest_burden    -9.479734  0.999944             False   
x25            loan_purpose_p4     1.988391       NaN             False   
x9   Neg_ammortization_not_neg     1.645580  0.999978             False   
x22            loan_purpose_p1     1.562609       NaN             False   
x23            loan_purpose_p2     1.533457       NaN             False   
x20     Security_Type_Indriect     1.524815  0.999989             False   
x19            loan_type_type2     1.384544  0.999990             False   
x24            loan_purpose_p3     1.375897       NaN             False   
x11                       term     1.346744  1.000000             False   
x21       Security_Type_direct     1.346744  1.000000             False   
x16       Credit_Worthiness_l1     1.263878       NaN             False   
x8   Neg_ammortization_ne

In [76]:
print(feature_names)
print(result.params)
print(result.pvalues)

['Intercept' 'loan_amount' 'rate_of_interest' 'term' 'LTV'
 'Upfront_charges' 'loan_interest_burden' 'loan_term_pressure'
 'Credit_Worthiness_l1' 'Credit_Worthiness_l2' 'loan_type_type1'
 'loan_type_type2' 'loan_type_type3' 'Security_Type_Indriect'
 'Security_Type_direct' 'loan_purpose_p1' 'loan_purpose_p2'
 'loan_purpose_p3' 'loan_purpose_p4' 'open_credit_nopc' 'open_credit_opc'
 'business_or_commercial_b/c' 'business_or_commercial_nob/c'
 'approv_in_adv_nopre' 'approv_in_adv_pre' 'Neg_ammortization_neg_amm'
 'Neg_ammortization_not_neg']
x1      0.307920
x2     -0.200739
x3      0.001913
x4      0.501742
x5     -0.293344
x6     -0.424961
x7     -0.087587
x8      1.263778
x9      1.645580
x10     0.944760
x11     1.346744
x12     0.617847
x13    12.389096
x14    -9.479734
x15     0.495841
x16     1.263878
x17     0.734694
x18     0.414936
x19     1.384544
x20     1.524815
x21     1.346744
x22     1.562609
x23     1.533457
x24     1.375897
x25     1.988391
x26     0.920957
dtype: float6

In [ ]:
print(feature_names)

In [47]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "../data/raw/Loan_Default.csv"

cols_to_ignore = ['LoanID']
target_col = 'Default'

label_encode_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
one_hot_encode_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']

MODEL_PATH = "loan_default_pipeline.pkl"

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(DATA_PATH)

# Drop ignored columns
df = df.drop(columns=cols_to_ignore)

X = df.drop(columns=[target_col])
y = df[target_col]

# Identify numerical columns
numerical_cols = [
    col for col in X.columns
    if col not in label_encode_cols + one_hot_encode_cols
]

# -----------------------------
# PREPROCESSING
# -----------------------------
label_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

one_hot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        ("label", label_encoder, label_encode_cols),
        ("onehot", one_hot_encoder, one_hot_encode_cols),
        ("num", StandardScaler(), numerical_cols)
    ]
)

# -----------------------------
# MODEL
# -----------------------------
model = LogisticRegression(max_iter=1000)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# -----------------------------
# TRAIN / TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# TRAIN
# -----------------------------
pipeline.fit(X_train, y_train)

# -----------------------------
# EVALUATE
# -----------------------------
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# -----------------------------
# SAVE PIPELINE
# -----------------------------
joblib.dump(pipeline, MODEL_PATH)
print(f"\nPipeline saved as {MODEL_PATH}")


Accuracy: 0.8852946935578617

Classification Report:
               precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.61      0.03      0.06      5931

    accuracy                           0.89     51070
   macro avg       0.75      0.52      0.50     51070
weighted avg       0.85      0.89      0.84     51070


Pipeline saved as loan_default_pipeline.pkl


In [48]:
import joblib
import pandas as pd

MODEL_PATH = "loan_default_pipeline.pkl"

# Load trained pipeline
pipeline = joblib.load(MODEL_PATH)

def predict_default(raw_input: dict):
    """
    raw_input: dictionary with raw values (same as dataset columns)
    """

    # Convert dict → DataFrame
    input_df = pd.DataFrame([raw_input])

    # Predict class
    prediction = pipeline.predict(input_df)[0]

    # Predict probability (if needed)
    probability = pipeline.predict_proba(input_df)[0][1]

    return {
        "Default_Prediction": int(prediction),
        "Default_Probability": round(probability, 4)
    }


# -----------------------------
# EXAMPLE USAGE
# -----------------------------
if __name__ == "__main__":
    sample_input = {
        "Age": 35,
        "Income": 60000,
        "LoanAmount": 250000,
        "CreditScore": 720,
        "MonthsEmployed": 60,
        "NumCreditLines": 5,
        "InterestRate": 11.5,
        "LoanTerm": 36,
        "DTIRatio": 0.35,

        "HasMortgage": "Yes",
        "HasDependents": "No",
        "HasCoSigner": "No",

        "Education": "Bachelor",
        "EmploymentType": "Salaried",
        "MaritalStatus": "Married",
        "LoanPurpose": "Home"
    }

    result = predict_default(sample_input)
    print(result)


{'Default_Prediction': 0, 'Default_Probability': np.float64(0.2703)}
